# 05 — Evaluation and RF Benchmark

CNN vs Random Forest with slide-level metrics.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / 'utils').exists():
        break
    ROOT = ROOT.parent
PHARMA = ROOT / 'projects' / 'spatial-pharma-dl'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(PHARMA))

from utils import st_helpers as st
st.set_seeds()
print('ROOT:', ROOT)
print('PHARMA:', PHARMA)


In [ ]:
import numpy as np
import pandas as pd
from src.data import load_config, pharma_outputs_dir
from src.labels import build_labels_cohort
from src.train import train_loso, loso_folds, _load_slide_data
from src.eval import evaluate_fold, train_eval_rf_baseline, save_benchmark_report

cfg = load_config()
oncology = cfg['cohorts']['oncology']
external = cfg['cohorts']['external']
labels = build_labels_cohort(oncology + external + cfg['cohorts']['benchmark'], cfg=cfg)
breast_labels = labels[labels['slide_id'].isin(oncology)]


In [ ]:
results = train_loso(oncology, breast_labels, cfg=cfg)
benchmark_rows = []
for r in results:
    ev = evaluate_fold(r)
    benchmark_rows.append({
        'model': 'cnn', 'fold': ev['fold'], 'val_slide': ev['val_slide'],
        'balanced_accuracy': ev['balanced_accuracy'], 'macro_f1': ev['macro_f1'],
        'mean_pearson_r': ev['mean_pearson_r'], 'mean_r2': ev['mean_r2'],
    })


In [ ]:
for fold, (train_slides, val_slide) in enumerate(loso_folds(oncology)):
    train_p, train_l = [], []
    for sid in train_slides:
        p, lab = _load_slide_data(sid, breast_labels)
        train_p.append(p); train_l.append(lab)
    val_p, val_l = _load_slide_data(val_slide, breast_labels)
    rf = train_eval_rf_baseline(np.concatenate(train_p), pd.concat(train_l), val_p, val_l, seed=0)
    benchmark_rows.append({
        'model': 'rf', 'fold': fold, 'val_slide': val_slide,
        'balanced_accuracy': rf['balanced_accuracy'], 'macro_f1': rf['macro_f1'],
        'mean_pearson_r': rf['mean_pearson_r'], 'mean_r2': rf['mean_r2'],
    })
save_benchmark_report(benchmark_rows)
pd.read_csv(pharma_outputs_dir() / 'benchmark_report.csv')


In [ ]:
from src.eval import predict_cnn
from src.patches import load_patch_arrays

for sid in external:
    try:
        patches, _ = load_patch_arrays(sid)
        predict_cnn(results[-1]['model'], patches[:50], device=results[-1]['device'])
        print(sid, 'external OK, spots sampled: 50')
    except FileNotFoundError as e:
        print(sid, e)


**Next:** `06_interpretability.ipynb`